# Temporal Robustness Stress Test

This notebook is Milestone 06 for the revised fraud-detection paper.

Research purpose:

- Compare optimistic random stratified evaluation with stricter chronological evaluation.
- Use only leakage-controlled strict feature sets.
- Quantify the Temporal Robustness Drop (TRD): chronological performance minus random-split performance.
- Save every result, checkpoint, log, table, and 600-DPI figure to Google Drive.

Interpretation rule:

- Negative TRD means the model becomes weaker under chronological deployment-style testing.
- This is not a leakage experiment. Leakage was Milestone 03. This milestone tests time-shift robustness.

In [ ]:
!pip -q install -U imbalanced-learn lightgbm xgboost seaborn
!pip -q install -q "pyarrow>=19,<24"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Drive screenshot showed: My Drive > Credit__ > Datasets
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')

DATASET_SEARCH_DIRS = [
    PROJECT_ROOT / 'Datasets',
    PROJECT_ROOT / '00_source_datasets',
    PROJECT_ROOT,
]

RUN_NAME = 'temporal_robustness_v1'
EXPERIMENT_ROOT = PROJECT_ROOT / '06_temporal_robustness' / RUN_NAME
CHECKPOINT_DIR = EXPERIMENT_ROOT / 'checkpoints'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
FIGURES_DIR = EXPERIMENT_ROOT / 'figures'
LOGS_DIR = EXPERIMENT_ROOT / 'logs'

for d in [EXPERIMENT_ROOT, CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 101, 202, 303, 404]
MODEL_NAMES = ['xgboost', 'lightgbm']

TRAIN_FRAC = 0.60
VAL_FRAC = 0.20
TEST_FRAC = 0.20

# Temporal caps use deterministic evenly spaced timeline sampling.
# Set any value to None if you want the full dataset.
CAPS = {
    'D1_creditcard': None,
    'D2_fraudTest': None,
    'D3_PaySim': 1_000_000,
}

SMOKE_TEST = False
SMOKE_CAP = 50_000

print('Project root:', PROJECT_ROOT)
print('Experiment root:', EXPERIMENT_ROOT)
print('Seeds:', SEEDS)

In [ ]:
import json
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    print('XGBoost unavailable:', exc)

try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print('LightGBM unavailable:', exc)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'legend.title_fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

def now():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def write_csv_atomic(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def append_log(message):
    line = f'[{now()}] {message}'
    print(line)
    with open(LOGS_DIR / 'run.log', 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def checkpoint_path(seed, dataset, split_policy, model_name):
    return CHECKPOINT_DIR / f'seed{seed}__{dataset}__{split_policy}__{model_name}.csv'

def is_done(seed, dataset, split_policy, model_name):
    return checkpoint_path(seed, dataset, split_policy, model_name).exists()

In [ ]:
# Task 2 raw prediction exports for paired bootstrap.
# This cell is intentionally independent from metric checkpoints.
import re

PREDICTION_EXPORT_DIR = EXPERIMENT_ROOT / 'prediction_exports'
PREDICTION_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def _safe_prediction_name(value):
    value = str(value)
    value = re.sub(r'[^A-Za-z0-9_.-]+', '_', value)
    return value.strip('_')

def prediction_export_path(seed, dataset, scenario, model_name, split='test'):
    file_name = (
        f"seed{int(seed)}__{_safe_prediction_name(dataset)}__{_safe_prediction_name(scenario)}__"
        f"{_safe_prediction_name(model_name)}__{_safe_prediction_name(split)}__predictions.csv"
    )
    return PREDICTION_EXPORT_DIR / file_name

def prediction_exports_exist(seed, dataset, model_name, scenarios, split='test'):
    return all(prediction_export_path(seed, dataset, scenario, model_name, split).exists() for scenario in scenarios)

def save_prediction_export(dataset, model_name, scenario, seed, split, y_true, scores, threshold, row_id=None):
    y_arr = np.asarray(y_true).reshape(-1)
    s_arr = np.asarray(scores).reshape(-1)

    if len(y_arr) != len(s_arr):
        raise ValueError(f'y_true length {len(y_arr)} != score length {len(s_arr)}')

    if row_id is None:
        row_arr = np.arange(len(y_arr))
    else:
        row_arr = np.asarray(row_id).reshape(-1)
        if len(row_arr) != len(y_arr):
            raise ValueError(f'row_id length {len(row_arr)} != y_true length {len(y_arr)}')

    out = pd.DataFrame({
        'dataset': str(dataset),
        'model': str(model_name),
        'scenario': str(scenario),
        'seed': int(seed),
        'split': str(split),
        'row_id': row_arr,
        'y_true': y_arr.astype(int),
        'score': s_arr.astype(float),
        'threshold': float(threshold),
    })

    path = prediction_export_path(seed, dataset, scenario, model_name, split)
    write_csv_atomic(path, out)
    append_log(f'Saved prediction export: {path}')
    return path

In [ ]:
def find_dataset_file(file_name):
    candidates = []
    for directory in DATASET_SEARCH_DIRS:
        candidates.append(directory / file_name)
        if directory.exists():
            candidates.extend(directory.glob(f'**/{file_name}'))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {file_name}')

DATASET_FILES = {
    'D1_creditcard': find_dataset_file('creditcard.csv'),
    'D2_fraudTest': find_dataset_file('fraudTest.csv'),
    'D3_PaySim': find_dataset_file('PS.csv'),
}

for name, path in DATASET_FILES.items():
    append_log(f'{name}: {path}')

write_json(EXPERIMENT_ROOT / 'dataset_file_manifest.json', {k: str(v) for k, v in DATASET_FILES.items()})

In [ ]:
def temporal_subsample(df, order_col, cap):
    if SMOKE_TEST:
        cap = min(SMOKE_CAP, len(df))
    if cap is None or len(df) <= cap:
        return df.reset_index(drop=True)
    positions = np.linspace(0, len(df) - 1, num=cap, dtype=np.int64)
    return df.iloc[positions].reset_index(drop=True)

def load_dataset(dataset_name):
    path = DATASET_FILES[dataset_name]
    df = pd.read_csv(path, low_memory=False)

    if dataset_name == 'D1_creditcard':
        df = df.drop_duplicates().reset_index(drop=True)
        label_col = 'Class'
        df['__order__'] = pd.to_numeric(df['Time'], errors='coerce')

    elif dataset_name == 'D2_fraudTest':
        label_col = 'is_fraud'
        df['__order__'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')

    elif dataset_name == 'D3_PaySim':
        label_col = 'isFraud'
        df = df.dropna(subset=[label_col]).reset_index(drop=True)
        df[label_col] = df[label_col].astype(int)
        df['__order__'] = pd.to_numeric(df['step'], errors='coerce')

    else:
        raise ValueError(dataset_name)

    df = df.dropna(subset=['__order__', label_col]).sort_values('__order__').reset_index(drop=True)
    df = temporal_subsample(df, '__order__', CAPS[dataset_name])
    fraud = int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum())
    append_log(f'{dataset_name}: rows={len(df):,}, fraud={fraud:,}, rate={fraud/len(df)*100:.4f}%')
    return df, label_col

def prepare_features(df, dataset_name):
    df = df.copy()

    if dataset_name == 'D1_creditcard':
        y = df['Class'].astype(int)
        X = df[[f'V{i}' for i in range(1, 29)] + ['Amount']].copy()
        return X, y

    if dataset_name == 'D2_fraudTest':
        df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
        df['dob_dt'] = pd.to_datetime(df['dob'], errors='coerce')
        df['trans_hour'] = df['trans_dt'].dt.hour
        df['trans_dayofweek'] = df['trans_dt'].dt.dayofweek
        df['trans_month'] = df['trans_dt'].dt.month
        df['age_years'] = (df['trans_dt'] - df['dob_dt']).dt.days / 365.25
        keep = ['category', 'amt', 'state', 'city_pop', 'trans_hour', 'trans_dayofweek', 'trans_month', 'age_years']
        return df[keep].copy(), df['is_fraud'].astype(int)

    if dataset_name == 'D3_PaySim':
        keep = ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']
        return df[keep].copy(), df['isFraud'].astype(int)

    raise ValueError(dataset_name)

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler(with_centering=False)),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=True)),
    ])
    return ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])

In [ ]:
def build_model(model_name, seed, scale_pos_weight=1.0):
    if model_name == 'xgboost':
        if xgb is None:
            raise RuntimeError('XGBoost not available')
        return xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='binary:logistic',
            eval_metric='aucpr',
            tree_method='hist',
            device='cuda',
            scale_pos_weight=scale_pos_weight,
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == 'lightgbm':
        if lgb is None:
            raise RuntimeError('LightGBM not available')
        return lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            class_weight='balanced',
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
    raise ValueError(model_name)

def get_scores(model, X):
    return model.predict_proba(X)[:, 1]

def choose_threshold(y_val, scores_val, c_fn=100, c_fp=1):
    best = None
    for tau in np.round(np.arange(0.01, 0.91, 0.01), 2):
        pred = (scores_val >= tau).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
        cost = c_fp * fp + c_fn * fn
        mcc = matthews_corrcoef(y_val, pred) if len(np.unique(pred)) > 1 else 0.0
        candidate = (cost, fn, fp, -mcc, tau)
        if best is None or candidate < best:
            best = candidate
    return float(best[-1])

def evaluate_predictions(y_true, scores, tau):
    pred = (scores >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    n = len(y_true)
    return {
        'threshold': float(tau),
        'n_test': int(n),
        'test_fraud': int(np.sum(y_true == 1)),
        'test_fraud_rate': float(np.mean(y_true == 1)),
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, pred)) if len(np.unique(pred)) > 1 else 0.0,
        'average_precision': float(average_precision_score(y_true, scores)),
        'roc_auc': float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': float(brier_score_loss(y_true, scores)),
        'false_alerts_per_10k': float(fp / n * 10_000),
        'missed_frauds_per_10k': float(fn / n * 10_000),
        'cost_per_10k': float((fp + 100 * fn) / n * 10_000),
    }

In [ ]:
def make_split_indices(y, split_policy, seed):
    y = pd.Series(y).reset_index(drop=True)
    n = len(y)
    idx = np.arange(n)

    if split_policy == 'RANDOM_STRATIFIED_SAFE':
        train_val_idx, test_idx = train_test_split(idx, test_size=TEST_FRAC, stratify=y, random_state=seed)
        y_train_val = y.iloc[train_val_idx]
        val_relative = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
        train_idx, val_idx = train_test_split(train_val_idx, test_size=val_relative, stratify=y_train_val, random_state=seed)
        return np.array(train_idx), np.array(val_idx), np.array(test_idx)

    if split_policy == 'CHRONOLOGICAL_SAFE':
        train_end = int(n * TRAIN_FRAC)
        val_end = int(n * (TRAIN_FRAC + VAL_FRAC))
        return idx[:train_end], idx[train_end:val_end], idx[val_end:]

    raise ValueError(split_policy)

def split_diagnostics(y, train_idx, val_idx, test_idx):
    def stats(name, subset_idx):
        yy = pd.Series(y).iloc[subset_idx]
        return {
            f'{name}_n': int(len(yy)),
            f'{name}_fraud': int(np.sum(yy == 1)),
            f'{name}_fraud_rate': float(np.mean(yy == 1)),
        }
    out = {}
    out.update(stats('train', train_idx))
    out.update(stats('val', val_idx))
    out.update(stats('test', test_idx))
    return out

def run_strict_pipeline(X, y, split_policy, model_name, seed, dataset_name=None):
    train_idx, val_idx, test_idx = make_split_indices(y, split_policy, seed)
    diagnostics = split_diagnostics(y, train_idx, val_idx, test_idx)
    if diagnostics['train_fraud'] == 0 or diagnostics['val_fraud'] == 0 or diagnostics['test_fraud'] == 0:
        raise RuntimeError(f'Zero-fraud temporal partition: {diagnostics}')

    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    preprocessor = make_preprocessor(X_train)
    X_train_t = preprocessor.fit_transform(X_train)
    X_val_t = preprocessor.transform(X_val)
    X_test_t = preprocessor.transform(X_test)

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, seed, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    val_scores = get_scores(model, X_val_t)
    tau = choose_threshold(y_val, val_scores)
    test_scores = get_scores(model, X_test_t)
    if dataset_name is not None:
        save_prediction_export(dataset_name, model_name, split_policy, seed, 'test', y_test, test_scores, tau, row_id=test_idx)
    metrics = evaluate_predictions(y_test, test_scores, tau)
    metrics['fit_seconds'] = float(fit_seconds)
    metrics.update(diagnostics)
    return metrics

In [ ]:
SPLIT_POLICIES = ['RANDOM_STRATIFIED_SAFE', 'CHRONOLOGICAL_SAFE']

def run_one(seed, dataset_name, split_policy, model_name):
    if is_done(seed, dataset_name, split_policy, model_name) and prediction_exports_exist(seed, dataset_name, model_name, [split_policy]):
        append_log(f'SKIP existing checkpoint and prediction export: seed={seed} {dataset_name} / {split_policy} / {model_name}')
        return pd.read_csv(checkpoint_path(seed, dataset_name, split_policy, model_name)).iloc[0].to_dict()

    df, label_col = load_dataset(dataset_name)
    X, y = prepare_features(df, dataset_name)
    metrics = run_strict_pipeline(X, y, split_policy, model_name, seed, dataset_name=dataset_name)

    row = {
        'run_name': RUN_NAME,
        'timestamp': now(),
        'seed': seed,
        'dataset': dataset_name,
        'split_policy': split_policy,
        'model': model_name,
        'feature_policy': 'strict',
        'smoke_test': SMOKE_TEST,
        'rows_used': int(len(df)),
        'fraud_rows_used': int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum()),
        'cap_used': CAPS[dataset_name],
        **metrics,
    }
    out = pd.DataFrame([row])
    write_csv_atomic(checkpoint_path(seed, dataset_name, split_policy, model_name), out)
    append_log(f'DONE seed={seed} {dataset_name} / {split_policy} / {model_name}: AP={row["average_precision"]:.4f}, MCC={row["mcc"]:.4f}')
    return row

all_rows = []
for seed in SEEDS:
    for dataset_name in ['D1_creditcard', 'D2_fraudTest', 'D3_PaySim']:
        for split_policy in SPLIT_POLICIES:
            for model_name in MODEL_NAMES:
                try:
                    row = run_one(seed, dataset_name, split_policy, model_name)
                    all_rows.append(row)
                    write_csv_atomic(RESULTS_DIR / 'partial_results_latest.csv', pd.DataFrame(all_rows))
                except Exception as exc:
                    append_log(f'ERROR seed={seed} {dataset_name} / {split_policy} / {model_name}: {repr(exc)}')
                    write_json(CHECKPOINT_DIR / f'ERROR__seed{seed}__{dataset_name}__{split_policy}__{model_name}.json', {
                        'seed': seed,
                        'dataset': dataset_name,
                        'split_policy': split_policy,
                        'model': model_name,
                        'error': repr(exc),
                        'timestamp': now(),
                    })

frames = [pd.read_csv(p) for p in sorted(CHECKPOINT_DIR.glob('seed*.csv'))]
results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
write_csv_atomic(RESULTS_DIR / 'temporal_robustness_results.csv', results)
results

In [ ]:
results = pd.read_csv(RESULTS_DIR / 'temporal_robustness_results.csv')

random_base = results[results['split_policy'] == 'RANDOM_STRATIFIED_SAFE'].copy()
random_base = random_base[[
    'seed', 'dataset', 'model',
    'average_precision', 'mcc', 'recall', 'precision', 'cost_per_10k', 'brier'
]].rename(columns={
    'average_precision': 'random_average_precision',
    'mcc': 'random_mcc',
    'recall': 'random_recall',
    'precision': 'random_precision',
    'cost_per_10k': 'random_cost_per_10k',
    'brier': 'random_brier',
})

trd = results.merge(random_base, on=['seed', 'dataset', 'model'], how='left')
trd['trd_average_precision'] = trd['average_precision'] - trd['random_average_precision']
trd['trd_mcc'] = trd['mcc'] - trd['random_mcc']
trd['trd_recall'] = trd['recall'] - trd['random_recall']
trd['trd_precision'] = trd['precision'] - trd['random_precision']
trd['cost_delta_per_10k'] = trd['cost_per_10k'] - trd['random_cost_per_10k']
trd['brier_delta'] = trd['brier'] - trd['random_brier']

write_csv_atomic(RESULTS_DIR / 'temporal_robustness_index.csv', trd)

summary = (
    trd[trd['split_policy'] == 'CHRONOLOGICAL_SAFE']
    .groupby(['dataset', 'model'])
    .agg(
        seeds=('seed', 'nunique'),
        random_ap_mean=('random_average_precision', 'mean'),
        chrono_ap_mean=('average_precision', 'mean'),
        trd_ap_mean=('trd_average_precision', 'mean'),
        trd_ap_std=('trd_average_precision', 'std'),
        random_mcc_mean=('random_mcc', 'mean'),
        chrono_mcc_mean=('mcc', 'mean'),
        trd_mcc_mean=('trd_mcc', 'mean'),
        trd_mcc_std=('trd_mcc', 'std'),
        cost_delta_mean=('cost_delta_per_10k', 'mean'),
        cost_delta_std=('cost_delta_per_10k', 'std'),
        chrono_test_fraud_rate_mean=('test_fraud_rate', 'mean'),
    )
    .reset_index()
)
write_csv_atomic(RESULTS_DIR / 'temporal_robustness_summary.csv', summary)
summary

In [ ]:
trd = pd.read_csv(RESULTS_DIR / 'temporal_robustness_index.csv')
plot_df = trd[trd['split_policy'] == 'CHRONOLOGICAL_SAFE'].copy()

dataset_labels = {'D1_creditcard': 'D1', 'D2_fraudTest': 'D2', 'D3_PaySim': 'D3'}
model_labels = {'xgboost': 'XGBoost', 'lightgbm': 'LightGBM'}
plot_df['dataset_label'] = plot_df['dataset'].map(dataset_labels)
plot_df['model_label'] = plot_df['model'].map(model_labels)

for metric, ylabel, filename in [
    ('trd_average_precision', 'Temporal robustness drop in AP', 'temporal_ap_drop_by_dataset'),
    ('trd_mcc', 'Temporal robustness drop in MCC', 'temporal_mcc_drop_by_dataset'),
    ('cost_delta_per_10k', 'Chronological cost delta per 10,000', 'temporal_cost_delta_by_dataset'),
]:
    fig, ax = plt.subplots(figsize=(7.0, 4.0))
    sns.barplot(
        data=plot_df,
        x='dataset_label',
        y=metric,
        hue='model_label',
        errorbar='sd',
        edgecolor='black',
        linewidth=0.45,
        ax=ax,
    )
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Dataset')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' across five seeds')
    ax.legend(title='Model', loc='best', frameon=True)
    ax.grid(axis='y', linestyle='--', linewidth=0.45, alpha=0.55)
    ax.grid(axis='x', visible=False)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    png = FIGURES_DIR / f'{filename}.png'
    pdf = FIGURES_DIR / f'{filename}.pdf'
    fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    append_log(f'Saved figure: {png}')

summary = pd.read_csv(RESULTS_DIR / 'temporal_robustness_summary.csv')
summary['row_label'] = summary['dataset'].map(dataset_labels) + ' - ' + summary['model'].map(model_labels)
heat = summary.pivot_table(index='row_label', values='trd_ap_mean', aggfunc='mean')

fig, ax = plt.subplots(figsize=(4.8, max(3.0, 0.45 * len(heat))))
sns.heatmap(
    heat,
    annot=True,
    fmt='.3f',
    cmap='vlag',
    center=0,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Chronological AP - random AP'},
    annot_kws={'fontsize': 8},
    ax=ax,
)
ax.set_title('Mean Temporal Robustness Drop in AP')
ax.set_xlabel('')
ax.set_ylabel('Dataset and model')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
fig.tight_layout()
png = FIGURES_DIR / 'temporal_ap_drop_heatmap.png'
pdf = FIGURES_DIR / 'temporal_ap_drop_heatmap.pdf'
fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf, bbox_inches='tight', facecolor='white')
plt.show()
append_log(f'Saved figure: {png}')

In [ ]:
summary = {
    'run_name': RUN_NAME,
    'completed_at': now(),
    'project_root': str(PROJECT_ROOT),
    'experiment_root': str(EXPERIMENT_ROOT),
    'seeds': SEEDS,
    'models': MODEL_NAMES,
    'split_policies': SPLIT_POLICIES,
    'caps': CAPS,
    'results_file': str(RESULTS_DIR / 'temporal_robustness_results.csv'),
    'index_file': str(RESULTS_DIR / 'temporal_robustness_index.csv'),
    'summary_file': str(RESULTS_DIR / 'temporal_robustness_summary.csv'),
    'figures_dir': str(FIGURES_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'n_result_rows': int(len(pd.read_csv(RESULTS_DIR / 'temporal_robustness_results.csv'))),
}
write_json(EXPERIMENT_ROOT / 'run_summary.json', summary)
summary

## Expected Runtime on Colab T4

Expected runtime is approximately **50-90 minutes**.

Why:

- Total jobs: 3 datasets x 2 split policies x 2 models x 5 seeds = 60 jobs.
- D3 PaySim is capped to 1,000,000 timeline-sampled rows by default.
- D1 and D2 run on the full cleaned datasets by default.

If Colab disconnects, rerun from the top. Existing checkpoint CSVs are skipped.